# A Renaissance Musical Cipher: Porta's *De Furtivis Literarum Notis* (1563)

## From the Bodleian's Shelves

The Bodleian Library holds two foundational Renaissance treatises on secret writing:

- **Johannes Trithemius**, *Polygraphiae libri sex* (1518) — the first printed book on cryptography, describing polyalphabetic substitution and the famous 'Ave Maria' cipher, where a sacred-sounding Latin prayer conceals a secret message word by word.
- **Giambattista della Porta**, *De Furtivis Literarum Notis* (1563, enlarged 1602) — 'On the Hidden Notes of Letters', a systematic survey of cipher methods that includes, uniquely, a **musical cipher**: letters encoded as musical notes.

> **A note on Trithemius**: despite often being cited alongside Porta in discussions of musical ciphers, Trithemius's own works contain no pitch-to-letter mapping. The musical ciphers sometimes attributed to him were actually developed later by Gustavus Selenus (1624), who cited Trithemius as inspiration. Porta's is the genuine Renaissance musical cipher.

---

## Porta's Musical Cipher

Porta's cipher maps the 22-letter Latin alphabet (I=J, U=V, no K or W) onto a **diatonic scale spanning an octave and a half** — from E3 up to A4, eleven steps.

The alphabet is split into two halves:

| Group | Letters | Note shape | Scale direction |
|:---:|:---:|:---:|:---:|
| First 11 | A B C D E F G H I L M | **Semibreve** (whole note ○) | Ascending E3→A4 |
| Last 11 | N O P Q R S T U X Y Z | **Minim** (half note 𝅗𝅥) | Descending A4→E3 |

The clever trick: the same pitch can encode **two different letters** — one from each half. The only thing that distinguishes them is the **note duration**. A G4 played as a whole note means **L**; the same G4 played as a half note means **O**.

The result looks and sounds like a real melody. That is the point: **steganography**, not just encryption. The message hides in plain hearing.

---

## Comparison with BACH and ABEGG

| Property | BACH (Bach, c.1740) | ABEGG (Schumann, 1830) | Porta (1563) |
|---|---|---|---|
| System | German note names | English note names | Scale position + duration |
| Disambiguator | Letter spelling | Letter spelling | **Note duration** |
| Alphabet covered | 8 letters (A–H) | 7 letters (A–G) | 22 letters (full Latin) |
| Purpose | Compositional signature | Dedication | Secret communication |
| Looks like music? | Yes, incidentally | Yes, incidentally | **Yes, by design** |

---
## Part 1: Implementing Porta's Cipher

In [ ]:
# Porta's cipher — De Furtivis Literarum Notis (1563 / 1602)
#
# 22-letter Latin alphabet mapped onto an 11-step diatonic scale.
# Starting pitch: E3 (MIDI 52), ascending by step to A4 (MIDI 69).
# Ascending half  → semibreve (whole note, 4 beats)
# Descending half → minim     (half note,  2 beats)
#
# Note on starting pitch: Porta wrote in staff notation without
# specifying an absolute pitch. E3 is the scholarly consensus
# (Eric Sams, Grove Music; Wikipedia Music cipher article) for
# how the cipher was received and transmitted. Some adaptations
# start on C. We follow the E3 consensus here.

# 11 diatonic scale steps: E F G A B C D E F G A
# (natural minor / Aeolian on E, or 'natural hexachord' extension)
# MIDI note numbers, starting E3 = 52
SCALE_PITCHES = [52, 53, 55, 57, 59, 60, 62, 64, 65, 67, 69]
#               E3  F3  G3  A3  B3  C4  D4  E4  F4  G4  A4

NOTE_NAMES = {
    0:'C', 1:'C#', 2:'D', 3:'Eb', 4:'E', 5:'F',
    6:'F#', 7:'G', 8:'Ab', 9:'A', 10:'Bb', 11:'B'
}

def midi_to_name(midi):
    octave = (midi // 12) - 1
    return f"{NOTE_NAMES[midi % 12]}{octave}"

# First 11 letters of the 22-letter Latin alphabet
# (I covers J; K omitted from Latin; ascending, semibreve)
ASCENDING_LETTERS  = list('ABCDEFGHILM')   # 11 letters

# Last 11 letters
# (U covers V; W omitted from Latin; descending from A4, minim)
DESCENDING_LETTERS = list('NOPQRSTUXY Z'.replace(' ', ''))  # 11 letters

assert len(ASCENDING_LETTERS)  == 11, f"Got {len(ASCENDING_LETTERS)}"
assert len(DESCENDING_LETTERS) == 11, f"Got {len(DESCENDING_LETTERS)}"
assert len(SCALE_PITCHES)      == 11

# Build encoding table: letter → (midi_pitch, duration_label, beats)
ENCODE = {}
for i, letter in enumerate(ASCENDING_LETTERS):
    ENCODE[letter] = (SCALE_PITCHES[i], 'semibreve', 4)

for i, letter in enumerate(DESCENDING_LETTERS):
    ENCODE[letter] = (SCALE_PITCHES[10 - i], 'minim', 2)  # descending

DECODE = {(pitch, dur): letter for letter, (pitch, dur, _) in ENCODE.items()}

# Normalisation: map letters outside the Latin 22 to their equivalents
NORMALISE = str.maketrans('JKVW', 'IIUU')

print("Porta cipher table:")
print()
print(f"  {'Letter':>8}  {'Pitch':>6}  {'Duration':>12}  {'Beats':>6}")
print("  " + "-" * 38)
for letter, (pitch, dur, beats) in sorted(ENCODE.items()):
    arrow = '↑' if dur == 'semibreve' else '↓'
    print(f"  {letter:>8}  {midi_to_name(pitch):>6}  {dur:>12}  {arrow} {beats}")

### The dual-use pitches

Because the ascending and descending halves share the same 11 pitches, each pitch encodes **two letters**. Only duration distinguishes them:

In [ ]:
print("Pitch   Ascending (semibreve)   Descending (minim)")
print("-" * 50)
for i in range(11):
    pitch = SCALE_PITCHES[i]
    asc  = ASCENDING_LETTERS[i]
    desc = DESCENDING_LETTERS[10 - i]
    name = midi_to_name(pitch)
    print(f"  {name:>4}       {asc}  (○ whole)           {desc}  (𝅗𝅥 half)")

print()
print("Every pitch encodes two letters. Duration is the only disambiguator.")

---
## Part 2: Encoding and Decoding a Message

In [ ]:
def porta_encode(text):
    """
    Encode a plaintext string using Porta's musical cipher.
    Returns a list of (letter, pitch, duration, beats) tuples.
    Spaces are kept as (" ", None, 'rest', 1) for a bar-line rest.
    """
    result = []
    for ch in text.upper().translate(NORMALISE):
        if ch in ENCODE:
            pitch, dur, beats = ENCODE[ch]
            result.append((ch, pitch, dur, beats))
        elif ch == ' ':
            result.append((' ', None, 'rest', 1))
        else:
            raise ValueError(f"'{ch}' cannot be encoded (Porta uses 22-letter Latin alphabet)")
    return result

def porta_decode(notes):
    """
    Decode a list of (pitch, duration) tuples back to plaintext.
    pitch=None means rest (word boundary).
    """
    result = []
    for pitch, dur in notes:
        if pitch is None:
            result.append(' ')
        else:
            result.append(DECODE.get((pitch, dur), '?'))
    return ''.join(result)

def show_encoding(text):
    encoded = porta_encode(text)
    print(f"Encoding: '{text}'")
    print()
    print(f"  {'Letter':>8}  {'Pitch':>6}  {'Duration':>12}  Direction")
    print("  " + "-" * 46)
    for letter, pitch, dur, beats in encoded:
        if pitch is None:
            print(f"  {'(space)':>8}  {'—':>6}  {'rest':>12}")
        else:
            direction = '↑ ascending' if dur == 'semibreve' else '↓ descending'
            print(f"  {letter:>8}  {midi_to_name(pitch):>6}  {dur:>12}  {direction}")
    print()
    # Decode back
    note_pairs = [(p, d) for (_, p, d, _) in encoded]
    recovered = porta_decode(note_pairs)
    print(f"  Decoded back: '{recovered}'")
    return encoded

# Encode a short message
show_encoding('PORTA')

In [ ]:
# Try a longer phrase
show_encoding('BODLEIAN')

---
## Part 3: The Steganographic Property — It Sounds Like Music

In [ ]:
%pip install midiutil --quiet

In [ ]:
from midiutil import MIDIFile

def write_porta_midi(text, filename, tempo=60):
    """
    Encode text using Porta's cipher and write to a MIDI file.
    Semibreves = 4 beats, minims = 2 beats (at the given tempo).
    """
    encoded = porta_encode(text)

    midi = MIDIFile(1)
    midi.addTempo(0, 0, tempo)

    time = 0
    for letter, pitch, dur, beats in encoded:
        if pitch is not None:
            midi.addNote(0, 0, pitch, time, beats * 0.9, 90)
        time += beats

    with open(filename, 'wb') as f:
        midi.writeFile(f)

    note_seq = [midi_to_name(p) if p else 'REST' for (_, p, d, _) in encoded]
    durs     = [d[:4] if p else '----' for (_, p, d, _) in encoded]
    print(f"Encoded '{text}' → {filename}")
    print(f"  Notes:     {note_seq}")
    print(f"  Durations: {durs}")

write_porta_midi('PORTA',    'porta_name.mid',    tempo=60)
print()
write_porta_midi('BODLEIAN', 'porta_bodleian.mid', tempo=60)

Open these `.mid` files and listen — the encoded messages sound like **slow, stepwise melodies**. Because the cipher maps letters alphabetically onto scale steps, sequences of nearby letters (like A B C D E) produce smooth ascending or descending runs, which are entirely plausible as real music.

This is the cipher's steganographic strength: it does not look or sound like a secret code. It looks like a composed melody.

---
## Part 4: Porta's Weakness — Alphabetical Order Betrays the Cipher

In [ ]:
# Porta's cipher has a structural weakness:
# letters are mapped IN ALPHABETICAL ORDER onto scale steps.
# A = lowest note, B = one step up, C = two steps up...
# This means: a rising scale = alphabetical sequence.

print("The structural weakness: alphabet order = scale order")
print()

# Encode the first few letters of the alphabet
test = 'ABCDE'
enc = porta_encode(test)
notes = [midi_to_name(p) for (_, p, d, _) in enc]
durs  = [d[:4] for (_, p, d, _) in enc]
print(f"  '{test}' → {notes}  ({durs})")
print(f"  All semibreves. A rising scale.")
print()

test2 = 'NOPQR'
enc2  = porta_encode(test2)
notes2 = [midi_to_name(p) for (_, p, d, _) in enc2]
durs2  = [d[:4] for (_, p, d, _) in enc2]
print(f"  '{test2}' → {notes2}  ({durs2})")
print(f"  All minims. A falling scale.")
print()
print("An adversary who knows the cipher system can read the message")
print("by noting: rising semibreves = early alphabet, falling minims = late alphabet.")
print()
print("Eric Sams (Grove Music, 1979) called this 'not a very strong method'.")
print("The steganographic cover is good; the cryptographic strength is poor.")

---
## Part 5: Searching for a Porta-Encoded Message in a MIDI File

In [ ]:
# Build a sample MIDI: a short melody with PORTA encoded inside,
# surrounded by filler notes.

from midiutil import MIDIFile

# Filler notes (diatonic, same scale — the melody blends in)
# Format: (midi_pitch, beats)
filler_before = [
    (57, 2), (59, 2), (60, 4),  # A3 B3 C4 (minims then semibreve)
    (62, 2), (60, 2),            # D4 C4
]

# PORTA encoded: P=F4/semi, O=G4/min, R=D4/min, T=B3/min, A=E3/semi
porta_enc = porta_encode('PORTA')
porta_notes = [(pitch, beats) for (_, pitch, _, beats) in porta_enc]

filler_after = [
    (64, 4), (62, 2), (60, 2),  # E4 D4 C4
    (59, 4), (57, 4),            # B3 A3
]

all_notes = filler_before + porta_notes + filler_after

midi = MIDIFile(1)
midi.addTempo(0, 0, 60)
time = 0
for pitch, beats in all_notes:
    midi.addNote(0, 0, pitch, time, beats * 0.9, 85)
    time += beats

with open('porta_hidden.mid', 'wb') as f:
    midi.writeFile(f)

print("Written: porta_hidden.mid")
print(f"Total notes: {len(all_notes)}")
print(f"PORTA is hidden at note positions {len(filler_before)}–{len(filler_before)+len(porta_notes)-1}")

In [ ]:
%pip install pretty_midi --quiet

In [ ]:
import pretty_midi

def extract_notes_with_duration(midi_file, beats_per_minute=60):
    """
    Extract (pitch, duration_label) pairs from a MIDI file.
    Maps note durations to 'semibreve' or 'minim' based on beat length.
    """
    pm = pretty_midi.PrettyMIDI(midi_file)
    spb = 60.0 / beats_per_minute  # seconds per beat

    notes = []
    for instrument in pm.instruments:
        if not instrument.is_drum:
            for note in instrument.notes:
                duration_beats = round((note.end - note.start) / spb)
                if duration_beats >= 3:
                    dur = 'semibreve'
                else:
                    dur = 'minim'
                notes.append((note.start, note.pitch, dur))
    notes.sort()
    return [(pitch, dur) for (_, pitch, dur) in notes]

def search_porta_motif(note_dur_sequence, target_text, label=None):
    """
    Search for a Porta-encoded word in a (pitch, duration) sequence.
    Returns positions of matches.
    """
    label = label or target_text
    target = [(pitch, dur) for (_, pitch, dur, _) in porta_encode(target_text)]
    n = len(target)
    matches = [i for i in range(len(note_dur_sequence) - n + 1)
               if note_dur_sequence[i:i+n] == target]
    print(f"Searching for '{label}':")
    print(f"  Target: {[(midi_to_name(p), d) for p,d in target]}")
    print(f"  Found {len(matches)} match(es) at positions: {matches}")
    return matches

# Extract notes from the hidden MIDI
seq = extract_notes_with_duration('porta_hidden.mid', beats_per_minute=60)
print(f"Extracted {len(seq)} notes from porta_hidden.mid")
print()

# Search for the hidden word
search_porta_motif(seq, 'PORTA')

---
## Part 6: Why Duration Matters — Contrast with BACH and ABEGG

In the BACH and ABEGG ciphers, **pitch alone** identifies the letter. In Porta's cipher, you need **both pitch and duration**. Let's see what happens if we search using pitch only (as we did for BACH):

In [ ]:
def search_pitch_only(note_dur_sequence, target_text, label=None):
    """
    Search using PITCH CLASS ONLY — ignoring duration.
    This is how the BACH/ABEGG search works.
    """
    label = label or target_text
    target_pitches = [pitch % 12 for (pitch, _) in
                      [(p, d) for (_, p, d, _) in porta_encode(target_text)]]
    seq_pitches    = [p % 12 for (p, _) in note_dur_sequence]
    n = len(target_pitches)
    matches = [i for i in range(len(seq_pitches) - n + 1)
               if seq_pitches[i:i+n] == target_pitches]
    print(f"Pitch-only search for '{label}':  {len(matches)} match(es) at {matches}")
    return matches

print("=" * 55)
print("Searching for PORTA in porta_hidden.mid")
print("=" * 55)
print()
print("1. Correct search (pitch + duration):")
m1 = search_porta_motif(seq, 'PORTA')
print()
print("2. Pitch-only search (ignoring duration, like BACH method):")
m2 = search_pitch_only(seq, 'PORTA')
print()
print("3. Now search for the 'shadow' word — same pitches, wrong durations:")
# The pitch sequence of PORTA is shared with MINOL (descending partner letters)
# Let's find what letters share those pitches
porta_enc = porta_encode('PORTA')
shadow_letters = []
for (letter, pitch, dur, beats) in porta_enc:
    # Find the other letter that uses the same pitch
    other_dur = 'minim' if dur == 'semibreve' else 'semibreve'
    other = DECODE.get((pitch, other_dur), '?')
    shadow_letters.append(other)
shadow = ''.join(shadow_letters)
print(f"   If we flip durations in PORTA, we'd read: '{shadow}'")
print(f"   (Same pitches, opposite durations — a completely different message)")

This is the key insight that separates Porta's cipher from BACH and ABEGG:

> **A pitch-only search will produce false positives** — it may find the right notes at the wrong durations, giving a different decoded message. You must always check both pitch *and* duration when working with Porta's cipher.

---
## Part 7: Three Ciphers, Three Centuries — Summary

In [ ]:
# Encode 'CAFE' in all three systems (letters A, C, E, F are common to all)
# to show how the same text sounds different

# BACH system (German)
GERMAN  = {'A':69,'B':70,'C':60,'D':62,'E':64,'F':65,'G':67,'H':71}
# ABEGG system (English)
ENGLISH = {'A':69,'B':71,'C':60,'D':62,'E':64,'F':65,'G':67}

test_word = 'CAFE'

german_notes  = [GERMAN[c]  for c in test_word]
english_notes = [ENGLISH[c] for c in test_word]
porta_enc     = porta_encode(test_word)
porta_notes   = [(midi_to_name(p), d[:4]) for (_, p, d, _) in porta_enc]

german_names  = [NOTE_NAMES[n % 12] for n in german_notes]
english_names = [NOTE_NAMES[n % 12] for n in english_notes]

print(f"Encoding '{test_word}' in three cipher systems:")
print()
print(f"  BACH  (German,  1740s):  {german_names}")
print(f"  ABEGG (English, 1830):   {english_names}")
print(f"  Porta (Latin,   1563):   {porta_notes}")
print()
print("Observations:")
print("  - German and English agree on C, E, F (no accidentals involved)")
print("  - Porta produces the same pitches BUT adds duration information")
print("  - Porta's C is E3 (lower octave) — it uses a different pitch range")
print("  - Only Porta's system can encode the full alphabet")

---
## Discussion Questions

1. Porta's cipher is **steganographic**: it hides the existence of a message, not just its content. The BACH and ABEGG ciphers are not steganographic in the same way — a listener who knows the system immediately recognises them. What does this distinction mean for the *purpose* of each cipher?

2. Porta's alphabetical ordering is a structural weakness: knowing the system breaks the cipher immediately. Is this a design flaw, or was the cipher intended primarily to deceive a listener who doesn't know a cipher exists at all?

3. Both Porta and Schumann were encoding **names**. Porta's system was for **secret communication**. Does the purpose of a cipher change how we analyse or decode it?

4. The Bodleian holds the physical copies of both Trithemius and Porta. What would you look for if you examined the original pages? (Hints: annotations, marginalia, staff notation, the layout of the cipher tables.)

5. We noted that Trithemius's works contain no musical cipher directly — the link is indirect, via Selenus (1624). How do we evaluate a source that says 'Trithemius used musical ciphers'? What does this tell us about how technical knowledge is transmitted and attributed over centuries?

---

## Further Reading

- Porta, G. della (1563). *De Furtivis Literarum Notis* — [1563 edition on Internet Archive](https://archive.org/details/bub_gb_sc-Zaq8_jFIC)
- Trithemius, J. (1518). *Polygraphiae libri sex* — digitised via the Bayerische Staatsbibliothek
- Sams, E. (1979). 'Musical cryptography'. *Grove Music Online*
- Code, D.L. (2022). 'Can musical encryption be both secure and musical?' *Cryptologia* 47(4)
- [Music cipher — Wikipedia](https://en.wikipedia.org/wiki/Music_cipher)

---
*Humanities Programming class — University of Oxford*